# AlphaDoc-Analytics: Exploratory Data Analysis & Predictive Modeling

This notebook demonstrates the underlying data science pipeline for the **AlphaDoc-Analytics** platform. We walk through: 
1. Ingesting structured financial quarterly time series.
2. Computing statistical metrics and Quarter-over-Quarter (QoQ) growth rates.
3. Modeling and forecasting upcoming performance indicators (Revenue, Operating Margin, Net Income) using Linear Regression.
4. Constructing prediction intervals (95% Confidence Intervals) based on residual variance.
5. Plotting results with visual accuracy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Set visual style
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries loaded successfully.")

## 1. Data Ingestion & Structuring
We instantiate a historical time series representing extracted quarterly reports.

In [ ]:
# Create structured historical DataFrame
data = {
    "Period": ["Q1 2025", "Q2 2025", "Q3 2025", "Q4 2025"],
    "Revenue": [1508.00, 1568.32, 1631.05, 1696.29],
    "Operating_Margin": [18.60, 19.00, 19.40, 19.80],
    "Net_Income": [218.92, 228.62, 238.75, 249.33],
    "EPS": [1.19, 1.25, 1.30, 1.36]
}

df = pd.DataFrame(data)
print("Historical Dataset:")
print(df)

## 2. Statistical Analysis: Growth and Metrics
We calculate Quarter-over-Quarter growth rates to identify trend patterns.

In [ ]:
# Compute QoQ growth percentage
df["Revenue_Growth_QoQ"] = df["Revenue"].pct_change() * 100
df["Net_Income_Growth_QoQ"] = df["Net_Income"].pct_change() * 100

# Clean NaNs resulting from pct_change shifts
df = df.fillna(0.0).round(2)

print("Enriched financial dataset:")
print(df)

## 3. Modeling & Trend Forecasting
We fit a Linear Regression model using the index of the quarter as our independent variable (time axis) and project it forward.

In [ ]:
# Setup independent variables
n_periods = len(df)
X = np.array(range(n_periods)).reshape(-1, 1)
y_rev = df["Revenue"].values

# Initialize and fit Linear Regression
model = LinearRegression()
model.fit(X, y_rev)

# Predict historical fits and project Q1 2026 (index 4)
y_pred_hist = model.predict(X)
next_period_idx = np.array([[n_periods]])
forecasted_rev = model.predict(next_period_idx)[0]

# Calculate Residual Standard Error to construct prediction intervals
residuals = y_rev - y_pred_hist
std_err = np.std(residuals) if len(residuals) > 1 else (y_rev[-1] * 0.05)
lower_bound = forecasted_rev - (1.96 * std_err)
upper_bound = forecasted_rev + (1.96 * std_err)

print(f"Linear Equation Model: Revenue = {model.coef_[0]:.2f} * Time_Index + {model.intercept_:.2f}")
print(f"Forecasted Revenue for Q1 2026: ${forecasted_rev:.2f} Million")
print(f"95% Confidence Interval Bounds: [${lower_bound:.2f}M - ${upper_bound:.2f}M]")

## 4. Evaluation Metrics
We calculate the Mean Absolute Error (MAE) and R-squared coefficient for the training sequence.

In [ ]:
mae = mean_absolute_error(y_rev, y_pred_hist)
r2 = r2_score(y_rev, y_pred_hist)

print(f"Training Mean Absolute Error: ${mae:.4f}M")
print(f"Training R-Squared Correlation: {r2:.4f} (Close to 1 indicates near-linear growth)")

## 5. Visualizing Historical and Forecast Trends

In [ ]:
# Prepare visualization sequence
periods_with_forecast = list(df["Period"]) + ["Q1 2026 (Forecast)"]
revenue_actuals = list(y_rev) + [None]
revenue_forecast = [None] * n_periods + [forecasted_rev]

# Regression line interpolation
regression_line = list(y_pred_hist) + [forecasted_rev]

plt.figure(figsize=(10, 5))
plt.bar(periods_with_forecast[:-1], revenue_actuals[:-1], color='indigo', alpha=0.5, label='Actual Revenue ($M)')
plt.plot(periods_with_forecast, regression_line, color='darkorange', marker='o', linewidth=2, label='Linear Regression Trend')
plt.bar(periods_with_forecast[-1], revenue_forecast[-1], color='crimson', alpha=0.7, label='Projected Revenue ($M)')

# Draw 95% Confidence Interval error bar over the projection
plt.errorbar(periods_with_forecast[-1], forecasted_rev, yerr=(1.96 * std_err), 
             fmt='none', ecolor='black', capsize=10, elinewidth=2.5, label='95% Confidence Bounds')

plt.title('AlphaDoc-Analytics: Revenue Actuals vs. Predictive Regressions')
plt.xlabel('Financial Period')
plt.ylabel('Revenue ($ Millions)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()